In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import itertools
import pandas as pd
import time
from scipy.interpolate import griddata

try:
    import matplotlib
    matplotlib.use('Agg')
except:
    pass

# ── Device ─────────────────────────────────────────────────────────────────
def get_preferred_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

device = get_preferred_device()
print(f"Using device: {device}")

# ── Exact Solution (1D) and Initial Condition (2D) ────────────────────────
def exact_solution_1d(x, t, D=0.01, R=1.0):
    """The traveling wave solution for the 1D Fisher-KPP equation."""
    sqrt_term  = np.sqrt(R / (2.0 * D))
    wave_speed = np.sqrt(2.0 * D * R)
    return 1.0 / (1.0 + np.exp(sqrt_term * (x - wave_speed * t)))

def initial_condition_2d(x, y):
    """A Gaussian-like initial condition for the 2D problem."""
    return np.exp(-((x - 0.5)**2 + (y - 0.5)**2) / 0.1)

# ── Activation helper ───────────────────────────────────────────────────────
class SinActivation(nn.Module):
    def forward(self, x):
        return torch.sin(x)

def get_activation(name):
    return {"tanh": nn.Tanh(), "sin": SinActivation(), "swish": nn.SiLU()}[name]

# ── PINN Model ──────────────────────────────────────────────────────────────
class PINN_FisherKPP_2D(nn.Module):

    def __init__(self, layers=(3, 50, 50, 50, 50, 50, 50, 50, 1),
                 activation_name="tanh", D=0.01, R=1.0,
                 adaptive_weights=True):
        super().__init__()
        self.D                = D
        self.R                = R
        self.activation_name  = activation_name
        self.adaptive_weights = adaptive_weights
        self.lambda_max       = 10_000.0
        self.lambda_ic        = 1.0
        self.lambda_bc        = 1.0
        self.lambda_res       = 1.0

        modules = []
        for i in range(len(layers) - 1):
            linear = nn.Linear(layers[i], layers[i + 1])
            nn.init.xavier_normal_(linear.weight)
            nn.init.zeros_(linear.bias)
            modules.append(linear)
            if i < len(layers) - 2:
                modules.append(get_activation(activation_name))
        self.net = nn.Sequential(*modules)

        self.loss_history      = []
        self.loss_ic_history   = []
        self.loss_bc_history   = []
        self.loss_res_history  = []
        self.l2_error_history  = []
        self.iteration_history = []
        self._global_iter      = 0

        n = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"  PINN layers={layers} | act={activation_name} | {n:,} params")

    def forward(self, x, y, t):
        return self.net(torch.cat([x, y, t], dim=1))

    def pde_residual(self, x_r, y_r, t_r):
        u    = self.forward(x_r, y_r, t_r)
        u_t  = torch.autograd.grad(u,   t_r, torch.ones_like(u),   create_graph=True)[0]
        u_x  = torch.autograd.grad(u,   x_r, torch.ones_like(u),   create_graph=True)[0]
        u_xx = torch.autograd.grad(u_x, x_r, torch.ones_like(u_x), create_graph=True)[0]
        u_y  = torch.autograd.grad(u,   y_r, torch.ones_like(u),   create_graph=True)[0]
        u_yy = torch.autograd.grad(u_y, y_r, torch.ones_like(u_y), create_graph=True)[0]
        return u_t - self.D * (u_xx + u_yy) - self.R * u * (1.0 - u)

    def compute_loss(self, x_ic, y_ic, t_ic, u_ic, x_bc, y_bc, t_bc, u_bc, x_r, y_r, t_r):
        l_ic  = torch.mean((self.forward(x_ic, y_ic, t_ic) - u_ic) ** 2)
        l_bc  = torch.mean((self.forward(x_bc, y_bc, t_bc) - u_bc) ** 2)
        l_res = torch.mean(self.pde_residual(x_r, y_r, t_r) ** 2)
        loss  = self.lambda_ic * l_ic + self.lambda_bc * l_bc + self.lambda_res * l_res
        return loss, l_ic, l_bc, l_res

    def _update_adaptive_weights(self, loss_ic, loss_bc, loss_res):
        eps            = 1e-10
        l_res          = loss_res.item()
        self.lambda_ic = min(l_res / (loss_ic.item() + eps), self.lambda_max)
        self.lambda_bc = min(l_res / (loss_bc.item() + eps), self.lambda_max)

    @torch.no_grad()
    def compute_l2_error(self, x_test, y_test, t_test, u_exact_np):
        # Note: L2 error is computed against a reference, not a true exact solution
        pred = self.forward(x_test, y_test, t_test).detach().cpu().numpy()
        if u_exact_np is None:
            return -1.0 # No exact solution available
        return np.linalg.norm(pred - u_exact_np) / np.linalg.norm(u_exact_np)

    def _record(self, it, loss, l_ic, l_bc, l_res, l2):
        self.loss_history.append(loss.item())
        self.loss_ic_history.append(l_ic.item())
        self.loss_bc_history.append(l_bc.item())
        self.loss_res_history.append(l_res.item())
        self.l2_error_history.append(l2)
        self.iteration_history.append(self._global_iter + it)

    def _make_scheduler(self, optimizer, schedule, iterations,
                        decay_rate=0.99, warmup_iters=1000):
        if schedule in ("exponential", "exponential_delayed"):
            return optim.lr_scheduler.ExponentialLR(optimizer, gamma=decay_rate)
        elif schedule == "cosine":
            return optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=iterations, eta_min=1e-7)
        elif schedule == "linear":
            return optim.lr_scheduler.LinearLR(
                optimizer, start_factor=1.0,
                total_iters=iterations, last_epoch=-1)
        else:
            print(f"  WARNING: Unknown schedule '{schedule}'")
            return None

    def train_adam(self, x_ic, y_ic, t_ic, u_ic,
                   x_bc, y_bc, t_bc, u_bc,
                   x_r_raw, y_r_raw, t_r_raw,
                   iterations=10_000,
                   learning_rate=1e-3,
                   decay_rate=0.99,
                   sched_step_every=100,
                   print_every=1000,
                   phase_name="Adam Training",
                   phase_label="phase1",
                   optimizer=None,
                   lr_schedule="cosine",
                   warmup_iters=1000,
                   x_test=None, y_test=None, t_test=None, u_ex=None):

        print(f"\n{'='*65}")
        print(f"  {phase_name}")
        print(f"  optimizer: Adam  lr={learning_rate}  schedule: {lr_schedule}")
        print(f"  iterations: {iterations}")
        print(f"{'='*65}")

        if x_test is None: # Create dummy test set if not provided
            x_np = np.linspace(0, 1, 51).astype(np.float32)
            y_np = np.linspace(0, 1, 51).astype(np.float32)
            x_mesh, y_mesh = np.meshgrid(x_np, y_np)
            x_flat = x_mesh.flatten().reshape(-1, 1)
            y_flat = y_mesh.flatten().reshape(-1, 1)
            t_flat = np.ones_like(x_flat)
            x_test = torch.from_numpy(x_flat).to(device)
            y_test = torch.from_numpy(y_flat).to(device)
            t_test = torch.from_numpy(t_flat).to(device)
            u_ex = None # No exact solution for 2D

        x_r = x_r_raw.clone().detach().requires_grad_(True)
        y_r = y_r_raw.clone().detach().requires_grad_(True)
        t_r = t_r_raw.clone().detach().requires_grad_(True)

        if optimizer is None:
            optimizer = optim.Adam(self.parameters(), lr=learning_rate)
        else:
            for pg in optimizer.param_groups:
                pg['lr'] = learning_rate

        scheduler = self._make_scheduler(optimizer, lr_schedule,
                                         iterations, decay_rate, warmup_iters)

        print(f"\n  {'Iter':>7}  {'Loss':>11}  {'L_IC':>11}  {'L_BC':>11}"
              f"  {'L_Res':>11}  {'L2_Err':>11}  {'LR':>9}  {'Time':>7}")
        print("  " + "-" * 90)

        t0 = time.time()

        for it in range(iterations):
            optimizer.zero_grad()
            loss, l_ic, l_bc, l_res = self.compute_loss(
                x_ic, y_ic, t_ic, u_ic, x_bc, y_bc, t_bc, u_bc, x_r, y_r, t_r)
            loss.backward()
            optimizer.step()

            if scheduler is not None:
                if lr_schedule == "exponential":
                    if (it + 1) % sched_step_every == 0:
                        scheduler.step()
                elif lr_schedule == "exponential_delayed":
                    if it >= warmup_iters and (it + 1) % sched_step_every == 0:
                        scheduler.step()
                else:
                    scheduler.step()

            if self.adaptive_weights and (it + 1) % 100 == 0:
                self._update_adaptive_weights(l_ic, l_bc, l_res)

            if it % print_every == 0 or it == iterations - 1:
                l2      = self.compute_l2_error(x_test, y_test, t_test, u_ex)
                cur_lr  = optimizer.param_groups[0]['lr']
                elapsed = time.time() - t0
                self._record(it, loss, l_ic, l_bc, l_res, l2)
                print(f"  {it:7d}  {loss.item():11.3e}  {l_ic.item():11.3e}"
                      f"  {l_bc.item():11.3e}  {l_res.item():11.3e}"
                      f"  {l2:11.3e}  {cur_lr:9.2e}  {elapsed:6.1f}s")

        self._global_iter += iterations
        final_l2 = self.compute_l2_error(x_test, y_test, t_test, u_ex)
        print(f"\n  ✓ Adam done — final L2 = {final_l2:.6e}")
        return final_l2, optimizer

    def train_lbfgs(self, x_ic, y_ic, t_ic, u_ic,
                    x_bc, y_bc, t_bc, u_bc,
                    x_r_raw, y_r_raw, t_r_raw,
                    outer_steps=500,
                    max_iter=20,
                    history_size=50,
                    tolerance_grad=1e-7,
                    tolerance_change=1e-9,
                    print_every=50,
                    phase_name="L-BFGS Fine-Tuning",
                    phase_label="phase3",
                    optimizer=None,
                    x_test=None, y_test=None, t_test=None, u_ex=None):

        print(f"\n{'='*65}")
        print(f"  {phase_name}")
        print(f"  outer_steps={outer_steps}  max_iter={max_iter}")
        print(f"{'='*65}")

        if x_test is None: # Create dummy test set if not provided
            x_np = np.linspace(0, 1, 51).astype(np.float32)
            y_np = np.linspace(0, 1, 51).astype(np.float32)
            x_mesh, y_mesh = np.meshgrid(x_np, y_np)
            x_flat = x_mesh.flatten().reshape(-1, 1)
            y_flat = y_mesh.flatten().reshape(-1, 1)
            t_flat = np.ones_like(x_flat)
            x_test = torch.from_numpy(x_flat).to(device)
            y_test = torch.from_numpy(y_flat).to(device)
            t_test = torch.from_numpy(t_flat).to(device)
            u_ex = None # No exact solution for 2D

        x_r = x_r_raw.clone().detach().requires_grad_(True)
        y_r = y_r_raw.clone().detach().requires_grad_(True)
        t_r = t_r_raw.clone().detach().requires_grad_(True)

        if optimizer is None:
            optimizer = optim.LBFGS(
                self.parameters(), lr=1.0, max_iter=max_iter,
                history_size=history_size,
                tolerance_grad=tolerance_grad,
                tolerance_change=tolerance_change,
                line_search_fn='strong_wolfe')

        cache = [None] * 4

        def closure():
            optimizer.zero_grad()
            loss, l_ic, l_bc, l_res = self.compute_loss(
                x_ic, y_ic, t_ic, u_ic, x_bc, y_bc, t_bc, u_bc, x_r, y_r, t_r)
            loss.backward()
            cache[:] = [loss, l_ic, l_bc, l_res]
            return loss

        print(f"\n  {'Step':>6}  {'Loss':>11}  {'L_IC':>11}  {'L_BC':>11}"
              f"  {'L_Res':>11}  {'L2_Err':>11}  {'Time':>7}")
        print("  " + "-" * 80)

        t0 = time.time()

        for step in range(outer_steps):
            optimizer.step(closure)

            if self.adaptive_weights and (step + 1) % 20 == 0:
                if cache[1] is not None:
                    self._update_adaptive_weights(cache[1], cache[2], cache[3])

            if step % print_every == 0 or step == outer_steps - 1:
                l2      = self.compute_l2_error(x_test, y_test, t_test, u_ex)
                elapsed = time.time() - t0
                self._record(step, cache[0], cache[1], cache[2], cache[3], l2)
                print(f"  {step:6d}  {cache[0].item():11.3e}"
                      f"  {cache[1].item():11.3e}  {cache[2].item():11.3e}"
                      f"  {cache[3].item():11.3e}  {l2:11.3e}  {elapsed:6.1f}s")

        self._global_iter += outer_steps * max_iter
        final_l2 = self.compute_l2_error(x_test, y_test, t_test, u_ex)
        print(f"\n  ✓ L-BFGS done — final L2 = {final_l2:.6e}")
        return final_l2, optimizer

    def save_checkpoint(self, path, optimizer=None):
        ck = {
            'state_dict'       : self.state_dict(),
            'lambda_ic'        : self.lambda_ic,
            'lambda_bc'        : self.lambda_bc,
            'lambda_res'       : self.lambda_res,
            'loss_history'     : self.loss_history,
            'loss_ic_history'  : self.loss_ic_history,
            'loss_bc_history'  : self.loss_bc_history,
            'loss_res_history' : self.loss_res_history,
            'l2_error_history' : self.l2_error_history,
            'iteration_history': self.iteration_history,
            'global_iter'      : self._global_iter,
        }
        if optimizer is not None:
            ck['optimizer_state'] = optimizer.state_dict()
        torch.save(ck, path)
        print(f"  Checkpoint saved -> {path}")

    def load_checkpoint(self, path, optimizer=None):
        ck = torch.load(path, map_location=next(self.parameters()).device,
                        weights_only=False)
        self.load_state_dict(ck['state_dict'])
        self.lambda_ic         = ck['lambda_ic']
        self.lambda_bc         = ck['lambda_bc']
        self.lambda_res        = ck['lambda_res']
        self.loss_history      = ck['loss_history']
        self.loss_ic_history   = ck['loss_ic_history']
        self.loss_bc_history   = ck['loss_bc_history']
        self.loss_res_history  = ck['loss_res_history']
        self.l2_error_history  = ck['l2_error_history']
        self.iteration_history = ck['iteration_history']
        self._global_iter      = ck.get('global_iter', 0)
        print(f"  Checkpoint loaded <- {path}")
        if optimizer is not None and 'optimizer_state' in ck:
            optimizer.load_state_dict(ck['optimizer_state'])
            print("    (optimizer state restored)")

    def plot_history(self, save_path='training_history.png', title=None):
        iters = np.array(self.iteration_history)
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        if title:
            fig.suptitle(title, fontsize=11)

        axes[0, 0].semilogy(iters, self.loss_history, 'b-', lw=1.5)
        axes[0, 0].set(xlabel='Iteration', ylabel='Total Loss',
                       title='Total Loss (log scale)')
        axes[0, 0].grid(alpha=0.3)

        axes[0, 1].plot(iters, self.l2_error_history, 'r-', lw=2, label='PINN (vs FDM)')
        axes[0, 1].set(xlabel='Iteration', ylabel='Relative L2 Error',
                       title='L2 Error vs FDM Reference')
        axes[0, 1].legend(fontsize=8)
        axes[0, 1].grid(alpha=0.3)

        axes[1, 0].semilogy(iters, self.loss_ic_history,  label='IC')
        axes[1, 0].semilogy(iters, self.loss_bc_history,  label='BC')
        axes[1, 0].semilogy(iters, self.loss_res_history, label='Residual')
        axes[1, 0].set(xlabel='Iteration', ylabel='Loss', title='Loss Components')
        axes[1, 0].legend()
        axes[1, 0].grid(alpha=0.3)

        axes[1, 1].plot(iters, self.l2_error_history, 'b-', lw=1.5)
        if np.any(iters >= 10_000):
            axes[1, 1].axvline(10_000, color='purple', ls=':', lw=1.5, label='Adam → L-BFGS')
        if np.any(iters >= 15_000):
            axes[1, 1].axvline(15_000, color='orange', ls='--', lw=1, label='Phase boundary')
        axes[1, 1].set(xlabel='Iteration', ylabel='L2 Error', title='Training Phases')
        axes[1, 1].legend()
        axes[1, 1].grid(alpha=0.3)

        plt.tight_layout()
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"  Plot saved -> {save_path}")


# ── Data Loading ────────────────────────────────────────────────────────────
def prepare_data_2d(path):
    d = np.load(path)
    print(f"\nDataset loaded from '{path}'")
    print(f"  Collocation : {d['collocation'].shape[0]:,}")
    print(f"  Initial     : {d['initial'].shape[0]:,}")
    print(f"  Boundary    : {d['boundary'].shape[0]:,}")

    def T(a):
        return torch.tensor(a, dtype=torch.float32)

    x_r  = T(d['collocation'][:, 0:1])
    y_r  = T(d['collocation'][:, 1:2])
    t_r  = T(d['collocation'][:, 2:3])

    x_ic = T(d['initial'][:, 0:1])
    y_ic = T(d['initial'][:, 1:2])
    t_ic = T(d['initial'][:, 2:3])

    x_bc = T(d['boundary'][:, 0:1])
    y_bc = T(d['boundary'][:, 1:2])
    t_bc = T(d['boundary'][:, 2:3])

    u_ic = T(initial_condition_2d(x_ic.numpy(), y_ic.numpy()))
    u_bc = torch.zeros_like(x_bc) # Assume zero Dirichlet boundary conditions

    return x_ic, y_ic, t_ic, u_ic, x_bc, y_bc, t_bc, u_bc, x_r, y_r, t_r

def load_fdm_reference(path, x_pinn, y_pinn, t_pinn):
    """Loads the FDM solution and interpolates it onto the PINN's test points."""
    try:
        fdm_data = np.load(path)
    except FileNotFoundError:
        print(f"Warning: FDM reference solution '{path}' not found. L2 error will be -1.")
        return None

    print(f"FDM reference loaded from '{path}'")
    x_fdm = fdm_data['x']
    y_fdm = fdm_data['y']
    u_fdm = fdm_data['u_exact']

    # The FDM data is on a grid. We need to interpolate it to the PINN's
    # (potentially scattered) test points.
    pinn_points = np.hstack((x_pinn, y_pinn))
    fdm_points = np.hstack((x_fdm, y_fdm))

    # We assume t is constant (t=1.0) for this comparison
    u_interpolated = griddata(fdm_points, u_fdm, pinn_points, method='cubic')

    # griddata can produce NaNs if points are outside the convex hull.
    # We will fill them with the nearest value.
    if np.any(np.isnan(u_interpolated)):
        u_interpolated_nearest = griddata(fdm_points, u_fdm, pinn_points, method='nearest')
        u_interpolated[np.isnan(u_interpolated)] = u_interpolated_nearest[np.isnan(u_interpolated)]

    return u_interpolated.reshape(-1, 1)


# ── Seeds & Data ────────────────────────────────────────────────────────────
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

x_ic, y_ic, t_ic, u_ic, x_bc, y_bc, t_bc, u_bc, x_r, y_r, t_r = prepare_data_2d('./data_samples_2d.npz')

x_ic = x_ic.to(device); y_ic = y_ic.to(device); t_ic = t_ic.to(device); u_ic = u_ic.to(device)
x_bc = x_bc.to(device); y_bc = y_bc.to(device); t_bc = t_bc.to(device); u_bc = u_bc.to(device)
x_r  = x_r.to(device);  y_r = y_r.to(device);   t_r  = t_r.to(device)

# Shared test points for L2 error calculation
x_np_test = np.linspace(0, 1, 51).astype(np.float32)
y_np_test = np.linspace(0, 1, 51).astype(np.float32)
x_mesh, y_mesh = np.meshgrid(x_np_test, y_np_test)
x_flat = x_mesh.flatten().reshape(-1, 1)
y_flat = y_mesh.flatten().reshape(-1, 1)
t_flat = np.ones_like(x_flat)
x_test_t = torch.from_numpy(x_flat).to(device)
y_test_t = torch.from_numpy(y_flat).to(device)
t_test_t = torch.from_numpy(t_flat).to(device)

# Load the FDM solution to use as the "exact" reference
u_ex_np = load_fdm_reference(
    '../results/2d/fdm_solution_2d.npz',
    x_pinn=x_flat, y_pinn=y_flat, t_pinn=t_flat
)

print(f"\nAll tensors on: {device}")


# ── Single best-config run (original workflow) ──────────────────────────────
pinn = PINN_FisherKPP_2D(
    layers=[3, 50, 50, 50, 50, 50, 50, 50, 1],
    activation_name="tanh",
    D=0.01, R=1.0,
    adaptive_weights=True,
).to(device)

train_data = dict(
    x_ic=x_ic, y_ic=y_ic, t_ic=t_ic, u_ic=u_ic,
    x_bc=x_bc, y_bc=y_bc, t_bc=t_bc, u_bc=u_bc,
    x_r_raw=x_r, y_r_raw=y_r, t_r_raw=t_r,
    x_test=x_test_t, y_test=y_test_t, t_test=t_test_t, u_ex=u_ex_np,
)

phase1_l2, _ = pinn.train_adam(
    **train_data,
    iterations=10_000, learning_rate=1e-3,
    lr_schedule="exponential", print_every=1000,
    phase_name="Phase 1 - Adam (10k iters, lr=1e-3)",
    phase_label="phase1",
)


phase2_l2, _ = pinn.train_adam(
    **train_data,
    iterations=5_000, learning_rate=1e-4,
    lr_schedule="exponential_delayed", print_every=1000,
    phase_name="Phase 2 - Adam Continue (5k iters, lr=1e-4)",
    phase_label="phase2",
)

phase3_l2, _ = pinn.train_lbfgs(
    **train_data,
    outer_steps=500, print_every=50,
    phase_name="Phase 3 - L-BFGS Fine-Tuning",
    phase_label="phase3",
)

print("\n" + "=" * 65)
print("  FINAL RESULTS — Single Run (2D vs FDM)")
print("=" * 65)
rows = [
    ("Phase 1 - Adam (10k)",     phase1_l2),
    ("Phase 2 - Adam (5k more)", phase2_l2),
    ("Phase 3 - L-BFGS",         phase3_l2),
]
for label, err in rows:
    print(f"  {label:<35} {err:.4e}")

pinn.plot_history(save_path='../results/2d/training_history_single_run_2d.png',
                  title='Single Run 2D: tanh | 7x50 | cosine/cosine/LBFGS')

np.savez('../results/2d/pinn_results_single_2d.npz',
         l2_phase1=phase1_l2, l2_phase2=phase2_l2, l2_phase3=phase3_l2,
         loss_history=pinn.loss_history,
         l2_error_history=pinn.l2_error_history,
         iteration_history=pinn.iteration_history)
print("\nSaved: ../results/2d/pinn_results_single_2d.npz")
raise SystemExit('Sweep skipped')


# ── Combination Sweep (2D) ───────────────────────────────────────
SWEEP_ITERS_P1   = 10_000
SWEEP_ITERS_P2   = 5_000
SWEEP_LBFGS      = 500
SWEEP_DECAY      = 0.99
SWEEP_STEP_EVERY = 100
SWEEP_WARMUP     = 1_000
SWEEP_PRINT      = 10_000   # only print at start/end of each phase

ACTIVATIONS   = ["tanh"]
LR_SCHEDULES  = ["exponential", "exponential_delayed", "cosine", "linear"]
ARCHITECTURES = [
    ([3, 50, 50, 50, 50, 50, 50, 50, 1], "7x50"),
    ([3, 100, 100, 100, 100, 100, 100, 1], "6x100"),
]

phase_schedule_combos = list(itertools.product(LR_SCHEDULES, repeat=2))  # 16 (only P1,P2)

all_combos = [
    (act, arch, arch_name, s1, s2)
    for act               in ACTIVATIONS
    for (arch, arch_name) in ARCHITECTURES
    for (s1, s2)          in phase_schedule_combos
]

print(f"\nTotal sweep runs : {len(all_combos)}")
print(f"  activations    : {ACTIVATIONS}")
print(f"  architectures  : {[a[1] for a in ARCHITECTURES]}")
print(f"  phase combos   : {len(phase_schedule_combos)}  (4^2 = 16)")

sweep_results = []
sweep_t0      = time.time()

for run_idx, (act, arch, arch_name, s1, s2) in enumerate(all_combos, 1):

    run_name = f"2d_{act}__{arch_name}__{s1}|{s2}|lbfgs"
    print(f"\n{'='*70}")
    print(f"[{run_idx}/{len(all_combos)}]  {run_name}")
    print(f"{'='*70}")

    torch.manual_seed(42)
    np.random.seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)

    model = PINN_FisherKPP_2D(
        layers=arch, activation_name=act,
        D=0.01, R=1.0, adaptive_weights=True,
    ).to(device)

    td = dict(
        x_ic=x_ic, y_ic=y_ic, t_ic=t_ic, u_ic=u_ic,
        x_bc=x_bc, y_bc=y_bc, t_bc=t_bc, u_bc=u_bc,
        x_r_raw=x_r, y_r_raw=y_r, t_r_raw=t_r,
        x_test=x_test_t, y_test=y_test_t, t_test=t_test_t, u_ex=u_ex_np,
    )

    t0 = time.time()

    l2_p1, opt = model.train_adam(
        **td, iterations=SWEEP_ITERS_P1, learning_rate=1e-3,
        decay_rate=SWEEP_DECAY, sched_step_every=SWEEP_STEP_EVERY,
        print_every=SWEEP_PRINT, warmup_iters=SWEEP_WARMUP,
        phase_name=f"Phase 1 ({s1})", phase_label="phase1",
        lr_schedule=s1,
    )

    l2_p2, opt = model.train_adam(
        **td, iterations=SWEEP_ITERS_P2, learning_rate=1e-4,
        decay_rate=SWEEP_DECAY, sched_step_every=SWEEP_STEP_EVERY,
        print_every=SWEEP_PRINT, warmup_iters=SWEEP_WARMUP,
        phase_name=f"Phase 2 ({s2})", phase_label="phase2", optimizer=opt,
        lr_schedule=s2,
    )

    l2_p3, _ = model.train_lbfgs(
        **td, outer_steps=SWEEP_LBFGS,
        print_every=SWEEP_LBFGS,   # only print final step
        phase_name="Phase 3 (L-BFGS)", phase_label="phase3",
    )

    elapsed = time.time() - t0

    # Save per-run L2 vs iteration plot
    plot_path = f"../results/2d/sweep/runs/l2_{run_name.replace('|','_')}.png"
    model.plot_history(
        save_path=plot_path,
        title=run_name,
    )

    sweep_results.append({
        "rank"        : 0,
        "run"         : run_name,
        "activation"  : act,
        "architecture": arch_name,
        "sched_p1"    : s1,
        "sched_p2"    : s2,
        "sched_p3"    : "lbfgs",
        "l2_phase1"   : float(l2_p1),
        "l2_phase2"   : float(l2_p2),
        "l2_phase3"   : float(l2_p3),
        "time_min"    : round(elapsed / 60, 2),
    })

    print(f"  ✓  P1={l2_p1:.3e}  P2={l2_p2:.3e}  P3={l2_p3:.3e}"
          f"  ({elapsed/60:.1f} min)"
          f"  [total elapsed: {(time.time()-sweep_t0)/60:.1f} min]")

# ── Results table ───────────────────────────────────────────────────────────
df = (pd.DataFrame(sweep_results)
        .sort_values("l2_phase3")
        .reset_index(drop=True))
df.index    += 1
df["rank"]   = df.index

# Reorder columns nicely
df = df[["rank", "activation", "architecture",
         "sched_p1", "sched_p2", "sched_p3",
         "l2_phase1", "l2_phase2", "l2_phase3",
         "time_min", "run"]]

print("\n" + "=" * 120)
print(f"  SWEEP RESULTS (2D) — all {len(all_combos)} combinations, ranked by Phase 3 L2 (best → worst)")
print("=" * 120)
print(df.to_string(index=False))

filename = f"../results/2d/sweep/sweep_results_2d_{len(all_combos)}.csv"
import os
os.makedirs("../results/2d/sweep", exist_ok=True)
df.to_csv(filename, index=False)
print(f"\nSaved: {filename}")

# ── Top-10 and bottom-5 summary ─────────────────────────────────────────────
print("\n── Top 10 runs (2D) ───────────────────────────────────────────────────")
print(df.head(10)[["rank", "activation", "architecture",
                    "sched_p1", "sched_p2",
                    "l2_phase1", "l2_phase2", "l2_phase3",
                    "time_min"]].to_string(index=False))

print("\n── Bottom 5 runs (2D) ───────────────────────────────────────────────────")
print(df.tail(5) [["rank", "activation", "architecture",
                    "sched_p1", "sched_p2",
                    "l2_phase1", "l2_phase2", "l2_phase3",
                    "time_min"]].to_string(index=False))

best = df.iloc[0]
print(f"\n{'='*60}")
print(f"  BEST RUN (2D)")
print(f"{'='*60}")
print(f"  Activation   : {best['activation']}")
print(f"  Architecture : {best['architecture']}")
print(f"  Schedules    : P1={best['sched_p1']}  P2={best['sched_p2']}  P3=L-BFGS")
print(f"  L2 Phase 1   : {best['l2_phase1']:.4e}")
print(f"  L2 Phase 2   : {best['l2_phase2']:.4e}")
print(f"  L2 Phase 3   : {best['l2_phase3']:.4e}")
print(f"  Time         : {best['time_min']:.2f} min")
print(f"\nTotal sweep time: {(time.time()-sweep_t0)/60:.1f} min")


Using device: cuda

Dataset loaded from './data_samples_2d.npz'
  Collocation : 100,000
  Initial     : 10,000
  Boundary    : 40,000
FDM reference loaded from '../results/2d/fdm_solution_2d.npz'

All tensors on: cuda
  PINN layers=[3, 50, 50, 50, 50, 50, 50, 50, 1] | act=tanh | 15,551 params

  Phase 1 - Adam (10k iters, lr=1e-3)
  optimizer: Adam  lr=0.001  schedule: exponential
  iterations: 10000

     Iter         Loss         L_IC         L_BC        L_Res       L2_Err         LR     Time
  ------------------------------------------------------------------------------------------


/home/aloo/.local/lib/python3.14/site-packages/torch/autograd/graph.py:841: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:270.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        0    2.100e-01    1.950e-01    3.519e-03    1.151e-02    7.094e-01   1.00e-03     0.4s
     1000    1.515e-03    1.884e-03    3.134e-04    5.047e-04    4.357e-02   9.04e-04   131.4s
     2000    1.944e-04    1.944e-03    3.028e-04    6.476e-05    3.798e-02   8.18e-04   262.5s
     3000    7.652e-05    1.741e-03    2.577e-04    2.549e-05    3.264e-02   7.40e-04   393.6s
     4000    4.017e-05    1.640e-03    2.531e-04    1.339e-05    3.221e-02   6.69e-04   524.7s
     5000    2.307e-05    1.601e-03    2.275e-04    7.687e-06    2.957e-02   6.05e-04   656.0s
     6000    1.671e-05    1.595e-03    2.025e-04    5.567e-06    2.698e-02   5.47e-04   787.2s
     7000    1.140e-05    1.686e-03    2.000e-04    3.798e-06    2.826e-02   4.95e-04   918.5s
     8000    1.366e-05    1.546e-03    1.764e-04    4.579e-06    2.257e-02   4.48e-04  1049.9s
     9000    1.230e-05    1.481e-03    1.628e-04    4.031e-06    2.182e-02   4.05e-04  1181.2s
     9999    6.530e-06    1.672e-03    1.671e-04  

SystemExit: Sweep skipped

/home/aloo/.local/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
